# M-03 Pose Estimation

Person Detection crop을 입력으로 받아 RTMPose-t의 COCO-17 keypoint를 추출하기 위한 baseline

## 현재 구현 범위

1. Colab 또는 macOS 로컬 경로 설정
2. 최소 실행 package 설치
3. ONNX Runtime 실행 provider 확인
4. OpenMMLab RTMPose-t ONNX 모델 무결성 확인
5. 추론 session 생성

Person Detection crop 입력과 keypoint 추론은 다음 작업에서 연결

# 1. 환경 설정

Colab에서는 Google Drive를 mount하고, 로컬에서는 현재 위치부터 Git 저장소 root를 탐색

In [1]:
import importlib.util
import os
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    DRIVE_MOUNT_POINT = Path("/content/drive")
    drive.mount(str(DRIVE_MOUNT_POINT))
    PROJECT_ROOT = (
        DRIVE_MOUNT_POINT / "MyDrive" / "Colab Notebooks" / "wardy-pose-fall"
    )
    runtime_name = "Google Colab"
else:
    def find_project_root(start_path: Path) -> Path:
        for candidate in (start_path, *start_path.parents):
            if (candidate / ".git").exists() and (candidate / "ml" / "notebook").is_dir():
                return candidate
        raise FileNotFoundError(
            "wardy 저장소 내부에서 Jupyter 실행 또는 WARDY_PROJECT_ROOT 설정 필요"
        )

    search_start = Path(
        os.environ.get("WARDY_PROJECT_ROOT", Path.cwd())
    ).expanduser().resolve()
    PROJECT_ROOT = find_project_root(search_start)
    runtime_name = "Local Jupyter"

REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"
MODEL_DIR = PROJECT_ROOT / "ml" / "checkpoints" / "rtmpose-t-onnx"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime: {runtime_name}")
print(f"Project repository: {PROJECT_ROOT.name}")
print(f"Requirements: {REQUIREMENTS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Model cache: {MODEL_DIR.relative_to(PROJECT_ROOT)}")

Runtime: Local Jupyter
Project repository: wardy-pose-fall
Requirements: requirements.txt
Model cache: ml/checkpoints/rtmpose-t-onnx


## 최소 package 설치

RTMPose ONNX 추론에 필요한 package만 설치. `mmcv`, `mmdet`, `mmpose`, PyTorch C++ extension compile은 사용하지 않음

In [2]:
%pip install --quiet --disable-pip-version-check -r {REQUIREMENTS_PATH}

Note: you may need to restart the kernel to use updated packages.


## ONNX Runtime provider 확인

Apple Silicon에서는 CoreML을 우선 사용하고, 지원하지 않는 연산은 CPU로 fallback

In [3]:
import platform
import sys
from importlib.metadata import version

import cv2
import numpy as np
import onnxruntime as ort

available_providers = ort.get_available_providers()
if platform.system() == "Darwin" and "CoreMLExecutionProvider" in available_providers:
    SESSION_PROVIDERS = ["CoreMLExecutionProvider", "CPUExecutionProvider"]
else:
    SESSION_PROVIDERS = ["CPUExecutionProvider"]

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Machine: {platform.machine()}")
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"ONNX Runtime: {version('onnxruntime')}")
print(f"Available providers: {available_providers}")
print(f"Selected providers: {SESSION_PROVIDERS}")

Python: 3.10.20
Platform: macOS-26.5.2-arm64-arm-64bit
Machine: arm64
NumPy: 1.26.4
OpenCV: 4.11.0
ONNX Runtime: 1.23.2
Available providers: ['CoreMLExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']
Selected providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider']


# 2. RTMPose-t baseline

OpenMMLab이 배포한 `RTMPose-t 256x192 COCO-17` ONNX archive를 내려받아 SHA-256 검증 후 추론 session 생성

In [4]:
import hashlib
from urllib.request import urlretrieve
from zipfile import ZipFile

MODEL_URL = (
    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/"
    "rtmpose-t_simcc-body7_pt-body7_420e-256x192-026a1439_20230504.zip"
)
ARCHIVE_PATH = MODEL_DIR / "rtmpose-t.zip"
MODEL_PATH = (
    MODEL_DIR
    / "20230831"
    / "rtmpose_onnx"
    / "rtmpose-t_simcc-body7_pt-body7_420e-256x192-026a1439_20230504"
    / "end2end.onnx"
)
ARCHIVE_SHA256 = "937003a70832d9cc34ea16927f504792f3133e92dda1b9c626236bbbe9e805cb"
MODEL_SHA256 = "a6c2f6a3896a4d51131d14d7a80a3d08b50f559af5a58a45d5b098aef510a70f"

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if ARCHIVE_PATH.exists():
    archive_state = "cached"
else:
    urlretrieve(MODEL_URL, ARCHIVE_PATH)
    archive_state = "downloaded"

archive_hash = sha256(ARCHIVE_PATH)
if archive_hash != ARCHIVE_SHA256:
    raise RuntimeError(f"Archive SHA-256 mismatch: {archive_hash}")

if not MODEL_PATH.exists():
    with ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(MODEL_DIR)

model_hash = sha256(MODEL_PATH)
if model_hash != MODEL_SHA256:
    raise RuntimeError(f"Model SHA-256 mismatch: {model_hash}")

pose_session = ort.InferenceSession(
    str(MODEL_PATH),
    providers=SESSION_PROVIDERS,
)

input_specs = [(item.name, item.shape, item.type) for item in pose_session.get_inputs()]
output_specs = [(item.name, item.shape, item.type) for item in pose_session.get_outputs()]

print(f"Model archive: {archive_state}")
print(f"Archive SHA-256: {archive_hash}")
print(f"Model SHA-256: {model_hash}")
print(f"Session providers: {pose_session.get_providers()}")
print(f"Input: {input_specs}")
print(f"Outputs: {output_specs}")
print("RTMPose-t session: ready")

Model archive: cached
Archive SHA-256: 937003a70832d9cc34ea16927f504792f3133e92dda1b9c626236bbbe9e805cb
Model SHA-256: a6c2f6a3896a4d51131d14d7a80a3d08b50f559af5a58a45d5b098aef510a70f
Session providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider']
Input: [('input', ['batch', 3, 256, 192], 'tensor(float)')]
Outputs: [('simcc_x', ['batch', 'MatMulsimcc_x_dim_1', 384], 'tensor(float)'), ('simcc_y', ['batch', 'MatMulsimcc_x_dim_1', 512], 'tensor(float)')]
RTMPose-t session: ready


2026-08-10 22:21:59.350 Python[33219:30845803] 2026-08-10 22:21:59.348756 [W:onnxruntime:, coreml_execution_provider.cc:113 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 5 number of nodes in the graph: 168 number of nodes supported by CoreML: 125


## 현재 checkpoint

RTMPose-t 모델 session 준비 완료. 다음 작업은 Person Detection의 사람별 crop을 `192x256` 입력으로 변환하고 COCO-17 keypoint를 복원하는 단계